# RMC-EXAFS from an EvAX project: copper foil

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nmarcella/feffGPU/blob/main/tutorials/03_rmc_exafs_from_evax.ipynb)

Reverse Monte Carlo (RMC) fitting moves the atoms of a model structure until its calculated EXAFS spectrum fits the
measured one. EvAX (J. Timoshenko and A. Kuzmin) is the established program for it, and the search of `rmc-exafs`
follows EvAX's rules. This notebook takes a finished EvAX calculation, a copper foil at the Cu K edge, and does three
things with it:

1. **Translate.** It reads EvAX's parameter file and writes the configuration of `rmc-exafs`, keyword by keyword.
2. **Check, in seconds, against the files EvAX wrote.** Is the misfit computed the same way? Does the same atomic
   structure give the same spectrum in both programs?
3. **Fit and compare.** It fits the same data from the same start with `rmc-exafs` and puts the two fitted
   structures side by side: the spectra, the misfits, the pair distributions and the numbers of the neighbor shells.

**You need the files of the EvAX example.** They are not part of this repository:

- the inputs, in a folder `in/`: `start.dat` (EvAX's parameter file), `Cu.p1` (the structure) and
  `foils_not_aligned.chi` (measured spectra; the fifth column is the copper foil);
- EvAX's outputs, in a folder `out/`: `EXAFS.dat`, `ipolEXAFS.dat`, `expWT.dat`, `wt.dat`, `partialEXAFS.dat`,
  `phantoms.xyz` and `phantoms_eq.xyz`.

EvAX itself is not needed and is not run.

**Time.** On the 12 GB GPU used for development: one minute for the potentials, seconds for the two checks, and
about 12 seconds for every step of the fit. The fit is the long part; its length is one number that you choose.
The notebook has not been timed on Colab.

**On Colab, choose a GPU first:** menu *Runtime → Change runtime type → T4 GPU*. Then run the cells from the top.

## 1. Install and check

On Colab, the next cell downloads the repository and installs the package. On your own computer it installs
nothing: install `feffgpu` first (`pip install -e .` in a checkout) and start Jupyter in the `tutorials` folder.

In [ ]:
import os
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB and not Path("feffGPU").exists():
    !git clone --depth 1 https://github.com/nmarcella/feffGPU.git
    !pip install -q ./feffGPU

In [ ]:
!feffgpu doctor

The last line must be `OK`. Without a GPU in the report, choose the GPU runtime on Colab and run the cells again.

## 2. The EvAX files

`EVAX_EXAMPLE` is the folder that holds `in/` and `out/`. Leave it empty on Colab to upload the two folders as
archives (`.tar` or `.zip`).

In [ ]:
import shutil
import tarfile
import zipfile

EVAX_EXAMPLE = os.environ.get("FEFFGPU_EVAX_EXAMPLE", "")      #@param {type:"string"}

if not EVAX_EXAMPLE:
    if not IN_COLAB:
        raise SystemExit("set EVAX_EXAMPLE to the folder that holds in/ and out/ of the EvAX example")
    from google.colab import files
    EVAX_EXAMPLE = "evax_example"
    Path(EVAX_EXAMPLE).mkdir(exist_ok=True)
    print("choose the archives of the folders in/ and out/")
    for name in files.upload():
        if name.endswith(".zip"):
            zipfile.ZipFile(name).extractall(EVAX_EXAMPLE)
        else:
            tarfile.open(name).extractall(EVAX_EXAMPLE, filter="data")

EVAX_IN, EVAX_OUT = Path(EVAX_EXAMPLE) / "in", Path(EVAX_EXAMPLE) / "out"
needed = [EVAX_IN / n for n in ("start.dat", "Cu.p1", "foils_not_aligned.chi")] + \
         [EVAX_OUT / n for n in ("EXAFS.dat", "ipolEXAFS.dat", "expWT.dat", "wt.dat", "partialEXAFS.dat", "phantoms.xyz", "phantoms_eq.xyz")]
missing = [str(p) for p in needed if not p.is_file()]
if missing:
    raise SystemExit("files of the EvAX example are missing: " + ", ".join(missing))

WORK = Path("cu_evax")                          # everything this notebook writes goes into this folder
WORK.mkdir(exist_ok=True)
for name in ("Cu.p1", "foils_not_aligned.chi"):
    shutil.copy(EVAX_IN / name, WORK / name)
print("EvAX inputs:", ", ".join(sorted(p.name for p in EVAX_IN.iterdir())))
print("EvAX outputs:", ", ".join(sorted(p.name for p in EVAX_OUT.iterdir())))

## 3. Read EvAX's parameter file

An EvAX parameter file is a list of keywords, each followed by its value.

In [ ]:
def read_evax_parameters(path):
    """The keywords of an EvAX parameter file with their values, as text."""
    parameters = {}
    for line in Path(path).read_text().splitlines():
        line = line.strip()
        if line and not line.startswith("#") and line not in ("BEGIN", "END"):
            key, _, value = line.partition(" ")
            parameters[key] = value.strip()
    return parameters


evax = read_evax_parameters(EVAX_IN / "start.dat")
for key in ("File_with_structure", "Supercell", "Initial_displacement", "Number_of_states", "Froze_in", "Stop_after",
            "Maximal_step_length", "Maximal_displacement", "Target_dicarding_rate", "Column_for_k", "Column_for_chi",
            "S02", "dE0", "Update_basis_every_...", "N_legs", "R_max_for_FEFF", "FEFF_criteria", "Edge", "Edge_type", "r_scf",
            "Space", "k_min", "k_max", "k_power", "R_min", "R_max", "R0_for_WT"):
    print(f"{key:28s} {evax[key]}")

## 4. Translate

Most keywords have a direct counterpart ([Configuration](https://github.com/nmarcella/feffGPU/blob/main/docs/rmc/configuration.md) names EvAX's keyword in
brackets for each of its own; every difference between the two programs is listed in
[Differences from EvAX](https://github.com/nmarcella/feffGPU/blob/main/docs/rmc/evax_differences.md)).

| EvAX | `rmc-exafs` | Note |
|---|---|---|
| `File_with_structure`, `Supercell 4 4 4` | `structure`, `format = "vasp"`, `supercell = [4, 4, 4]` | The `.p1` file is a VASP structure file with extra columns. 256 atoms. |
| `File_with_experimental_EXAFS_signal`, `Column_for_k 1`, `Column_for_chi 5` | `chi`, `chi_columns = [0, 4]` | Columns count from zero here. |
| `Edge Cu`, `Edge_type K` | `absorber = "Cu"`; `EDGE K` in the input of the potentials | |
| `R_max_for_FEFF 6.5` | `rpath = 6.5` | The cell (14.46 Å) is more than twice as wide, as required. |
| `N_legs 8` | `nleg = 4` | Four legs is the largest here. EvAX's own list of paths for this run has none with more (step 6). |
| `FEFF_criteria 1.5` | `criteria = 1.5` | |
| `r_scf -1` | `SCF 4.4275 0 30 0.1 1` in the input of the potentials | **Not "no self-consistency"**: see below. |
| `S02 0.9` | `[amplitude] s02 = 0.9`, held | |
| `dE0 -3.6` | `[amplitude] de0 = 3.6`, held | **The sign changes**: see below. |
| `Space w`, `k_min 3`, `k_max 13`, `k_power 2`, `R_min 1`, `R_max 6`, `R0_for_WT 3.5` | `objective = "wavelet"`, `wavelet_r0 = 3.5`, `window = { kmin = 3, kmax = 13, dk = 0, kweight = 2, rmin = 1, rmax = 6 }` | In wavelet space EvAX uses the range from `k_min` to `k_max` without a taper: `dk = 0` (checked in step 6). |
| `Number_of_states 64` | `[search] states = 64` | |
| `Froze_in 3000` | `froze_in = 3000` | |
| `Stop_after -1` | `generations = ...` | EvAX runs until it is stopped; here the number of generations is stated. |
| `Maximal_step_length 0.001` | `step = 0.001` | |
| `Target_dicarding_rate 0.5` | `acceptance = 0.5` | An older spelling of `Acceptance_rate`: a discarding rate of 0.5 is read as an acceptance rate of 0.5. |
| `Initial_displacement 0.05` | `start_kick = 0.05` | |
| `Update_basis_every_... 3` | `paths_every = 3` | |
| `Potential_type box`, `Maximal_displacement 0.4` | `[constraints] max_displacement = 0.4` | |
| `Maximal_number_of_clusters`, `Clustering_precision`, `Expansion_accuracy`, `Pathfinder_robustness` | none | EvAX groups similar paths and computes representatives. Here every path of every atom is computed. |
| `MT_seed`, `Number_of_threads`, `time1`, `time2`, `File_for_...` | `[run] seed`, none | The output files have fixed names here. |

Three of these need care.

**`r_scf -1` means that EvAX chooses the radius.** A negative value does not turn the self-consistency of the
potentials off. EvAX then takes the distance of the third neighbor shell as the radius, here 4.43 Å, and always
runs FEFF with self-consistent potentials. Potentials without self-consistency put the energy origin about 2 eV
elsewhere for copper, which is as large as the energy shift that is being translated.

**`dE0` changes sign, and the amplitude factor moves to the other side.** EvAX applies S0² and the energy shift to
the measured spectrum: it divides the data by S0² and reads them at `k' = sqrt(k² - 0.2625 dE0)`. This package
leaves the data alone and applies both to the calculated spectrum (the convention of xraylarch's `feffit`):
`S0² chi(k')` with `k' = sqrt(k² - 0.2625 de0)`. The same alignment is therefore `de0 = -dE0`. One difference
remains. EvAX multiplies the data by k² before it shifts them, so its target carries the factor `(k'/k)²`
relative to shifted data: 10 % at k = 3 and 0.6 % at k = 13 for this shift. Step 6 rebuilds EvAX's target from
the measured file to show it.

**The energy shift belongs to the potentials it was found with.** It aligns the measured spectrum with one
calculation. With other potentials it is another number. That is why the potentials below follow EvAX's FEFF
settings.

In [ ]:
supercell = [int(v) for v in evax["Supercell"].split()]
k_column, chi_column = int(evax["Column_for_k"]) - 1, int(evax["Column_for_chi"]) - 1
s02, de0 = float(evax["S02"]), -float(evax["dE0"])
rpath, criteria = float(evax["R_max_for_FEFF"]), float(evax["FEFF_criteria"])
nleg = min(int(evax["N_legs"]), 4)
kmin, kmax, kweight = float(evax["k_min"]), float(evax["k_max"]), int(float(evax["k_power"]))
rmin, rmax = float(evax["R_min"]), float(evax["R_max"])

SAMPLE = f"""[[sample]]
label = "Cu_foil"
structure = "Cu.p1"
format = "vasp"
supercell = {supercell}

  [[sample.edge]]
  name = "Cu"
  absorber = "{evax["Edge"]}"
  chi = "foils_not_aligned.chi"
  chi_columns = [{k_column}, {chi_column}]
  phase = "phase_Cu.pkl"
  rpath = {rpath}
  nleg = {nleg}
  criteria = {criteria}
"""

search_toml = f"""# EvAX's start.dat, keyword by keyword: the population search under EvAX's rules
[run]
output = "search"
seed = 1

{SAMPLE}  window = {{ kmin = {kmin}, kmax = {kmax}, dk = 0.0, kweight = {kweight}, rmin = {rmin}, rmax = {rmax} }}
  objective = "wavelet"
  wavelet_r0 = {float(evax["R0_for_WT"])}

[amplitude]
mode = "fixed"
s02 = {s02}
de0 = {de0}

[search]
states = {int(evax["Number_of_states"])}
generations = {int(evax["Froze_in"])}
froze_in = {int(evax["Froze_in"])}
step = {float(evax["Maximal_step_length"])}
acceptance = {float(evax["Target_dicarding_rate"])}
start_kick = {float(evax["Initial_displacement"])}
paths_every = {int(evax["Update_basis_every_..."])}
gradient_steps = 0

[constraints]
max_displacement = {float(evax["Maximal_displacement"])}
"""
(WORK / "search.toml").write_text(search_toml)
print(search_toml)

This file is the literal translation: the population search of `rmc-exafs` under EvAX's rules, with EvAX's numbers.
Step 7 says what it costs and which route this notebook takes instead.

## 5. The potentials

Both programs compute FEFF's scattering potentials once, for the ideal structure, and keep them for the whole fit.
The next cell cuts a cluster of the ideal copper lattice, applies EvAX's rule for the radius of the
self-consistency loop, and writes the FEFF input with the cards EvAX writes. `feffgpu phase` then computes the
potentials and the phase shifts on the GPU and stores them in one file, the phase file.

In [ ]:
import numpy as np
from ase.io import read
from feffgpu.md.potentials import cut_cluster, reference_input

ideal = read(WORK / "Cu.p1", format="vasp") * tuple(supercell)
CELL = float(ideal.cell.lengths()[0])
positions, symbols, distance = cut_cluster(ideal.positions, np.array(ideal.get_chemical_symbols()), np.array(ideal.cell),
                                           center=0, radius=7.0)

# EvAX's rule for r_scf < 0: the first shell; the next one at least 1 A farther; then the next one at least
# a tenth of the first distance farther than that. Its distance is the radius.
shells = np.unique(np.round(distance[1:], 6))
second = shells[shells - shells[0] >= 1.0][0]
scf_radius = float(shells[shells - second >= 0.1 * shells[0]][0])
print(f"neighbor shells (Å): {shells[:5]}  ->  radius of the self-consistency loop: {scf_radius:.4f} Å")

feff_input, _ = reference_input(positions, symbols, evax["Edge"], edge=evax["Edge_type"], scf_radius=0.0, phase_kmax=20.0,
                                rpath=rpath, title="ideal fcc Cu, the FEFF settings of EvAX",
                                cards=[f"SCF {scf_radius:.6f} 0 {evax['n_scf']} {evax['ca']} {evax['n_mix']}", "EXCHANGE 0 0 0"])
(WORK / "cu_cluster.inp").write_text(feff_input)
print("\n".join(feff_input.splitlines()[:13]), "\n ...")

In [ ]:
!feffgpu phase cu_evax/cu_cluster.inp cu_evax/phase_Cu.pkl

## 6. Two checks against EvAX's files

Before any fit, two questions can be answered in seconds, because EvAX wrote its final spectrum, its target and its
final atoms to files.

### Is the misfit the same number?

EvAX fits in wavelet space: the spectrum is resolved in k and in R at once. Its misfit is the square root of the
summed squared difference of the two wavelet transforms over the summed square of the target's. `expWT.dat` and
`wt.dat` hold EvAX's transforms of its target and of its final spectrum, so EvAX's final misfit can be computed
from them. The same number is then computed from the two spectra (`ipolEXAFS.dat`, `EXAFS.dat`) with this
package's own wavelet objective.

In [ ]:
import torch
from feffgpu.exafs import ETOK, FitWindow, k_grid
from feffgpu.exafs.wavelet import Wavelet

k = k_grid(20.0)                                # the k grid of the package: 0 to 20 1/Å in steps of 0.05


def on_grid(path):
    """A two-column spectrum that is on a 0.05 1/Å grid, placed on k (zero elsewhere)."""
    table = np.loadtxt(path)
    spectrum = np.zeros_like(k)
    index = np.round(table[:, 0] / 0.05).astype(int)
    spectrum[index[index < len(k)]] = table[index < len(k), 1]
    return spectrum


def evax_wavelet_file(path):
    """An EvAX wavelet file: R points and the complex transform (R, k)."""
    table = np.loadtxt(path)
    r = np.unique(table[:, 1])
    return r, (table[:, 2] + 1j * table[:, 3]).reshape(len(r), -1)


evax_target, evax_fit = on_grid(EVAX_OUT / "ipolEXAFS.dat"), on_grid(EVAX_OUT / "EXAFS.dat")
r_wt, wt_target = evax_wavelet_file(EVAX_OUT / "expWT.dat")
_, wt_fit = evax_wavelet_file(EVAX_OUT / "wt.dat")
inside = (r_wt >= rmin - 1e-9) & (r_wt <= rmax + 1e-9)
from_files = np.sqrt((np.abs(wt_fit[inside] - wt_target[inside]) ** 2).sum() / (np.abs(wt_target[inside]) ** 2).sum())

WAVELET_WINDOW = FitWindow(kmin=kmin, kmax=kmax, dk=0.0, kweight=kweight, rmin=rmin, rmax=rmax)
wavelet = Wavelet.for_window(k, WAVELET_WINDOW, r0=float(evax["R0_for_WT"]), device="cpu")
wavelet.set_target(evax_target)


def wavelet_misfit(spectrum):
    """EvAX's misfit of a calculated spectrum against EvAX's target, with this package's wavelet objective."""
    return float(wavelet.residual(torch.as_tensor(spectrum, dtype=torch.float32)[None])[0]) ** 0.5


print(f"EvAX's final misfit, from EvAX's own wavelet files:   {from_files:.4f}")
print(f"the same two spectra through this package's objective: {wavelet_misfit(evax_fit):.4f}")

The two numbers agree to about one percent of their value: `objective = "wavelet"` is EvAX's misfit. (This package
reports the ratio itself, which is the square of EvAX's number.)

EvAX's target can also be rebuilt from the measured file, which shows how EvAX uses S0² and the energy shift:

In [ ]:
measured = np.loadtxt(EVAX_IN / "foils_not_aligned.chi")
k_data, chi_data = measured[:, k_column], measured[:, chi_column]
k_target = np.loadtxt(EVAX_OUT / "ipolEXAFS.dat")[:, 0]
k_shifted = np.sqrt(k_target ** 2 - float(evax["dE0"]) * ETOK)          # where EvAX reads the data
rebuilt = np.interp(k_shifted, k_data, k_data ** kweight * chi_data) / k_target ** kweight / float(evax["S02"])
difference = np.abs(rebuilt - np.loadtxt(EVAX_OUT / "ipolEXAFS.dat")[:, 1]).max()
print(f"EvAX's target rebuilt from the measured file: largest difference {difference:.1e} (the file has 7 digits)")

### Does the same structure give the same spectrum?

`phantoms.xyz` holds the atoms of EvAX's final structure, `EXAFS.dat` the spectrum EvAX calculated for them, and
`partialEXAFS.dat` its parts by scattering path. The next cell computes the spectrum of the same atoms with this
package and compares, with the R-factor: the summed squared difference of the Fourier transforms of two spectra
inside the R window, divided by the summed square of one of them (xraylarch's definition; 0 means identical).

In [ ]:
from feffgpu import load_phase
from feffgpu.exafs import r_factor
from feffgpu.rmc import GpuRmc

phase = load_phase(WORK / "phase_Cu.pkl")
evax_atoms = np.loadtxt(EVAX_OUT / "phantoms.xyz", skiprows=2, usecols=(1, 2, 3))[: len(ideal)]
evax_sites = np.loadtxt(EVAX_OUT / "phantoms_eq.xyz", skiprows=2, usecols=(1, 2, 3))[: len(ideal)]

# the forward model in the frame of the calculation: no energy shift, no S0^2
forward = GpuRmc(phase, k, evax_fit, 0.0, CELL, window=(kmin + 0.5, kmax - 0.5), r_range=(rmin, rmax), rmax=rpath,
                 critpw=criteria, r_min=1.5, dk=1.0)


def spectrum_of(atoms):
    """The calculated spectrum of a structure (copies, atoms, 3), all paths, and its single-scattering part."""
    atoms = np.asarray(atoms, float).reshape(-1, len(ideal), 3)
    total, _ = forward.spectrum_only(atoms)
    single = forward.pairs.single_scattering(torch.as_tensor(atoms, device=forward.device, dtype=torch.float32))
    return total.cpu().numpy(), single.cpu().numpy()


def rf(model, reference):
    return r_factor(k, reference, model, kmin=kmin + 0.5, kmax=kmax - 0.5, dk=1.0, rmin=rmin, rmax=rmax, kweight=kweight)


ours_total, ours_single = spectrum_of(evax_atoms)
header = (EVAX_OUT / "partialEXAFS.dat").read_text().splitlines()[0]
legs = np.array([int(v) for v in header.split()[1:]])
partial = np.loadtxt(EVAX_OUT / "partialEXAFS.dat")
evax_single = np.zeros_like(k)
evax_single[np.round(partial[:, 0] / 0.05).astype(int)] = partial[:, 1:][:, legs == 2].sum(1)

print(f"EvAX's list of paths: {len(legs)} kinds, with {sorted(set(legs.tolist()))} legs")
print(f"R-factor between the two programs for the same atoms, all paths:        {rf(ours_total, evax_fit):.1e}")
print(f"                                                single scattering only: {rf(ours_single, evax_single):.1e}")
print(f"for scale, the R-factor of leaving multiple scattering out:             {rf(ours_single, ours_total):.1e}")

In [ ]:
import matplotlib.pyplot as plt
from feffgpu.exafs import ft, r_grid, window_tensor

BLUE, ORANGE, AQUA, GRAY, INK = "#2a78d6", "#eb6834", "#1baf7a", "#8a8984", "#0b0b0b"
FT_WINDOW = FitWindow(kmin=kmin + 0.5, kmax=kmax - 0.5, dk=1.0, kweight=kweight, rmin=rmin, rmax=rmax)
plt.rcParams.update({"axes.spines.top": False, "axes.spines.right": False, "axes.grid": True,
                     "grid.color": "#e4e3df", "legend.frameon": False, "figure.dpi": 110})


def chi_r(spectrum):
    """R (Å) and |chi(R)| of a spectrum on k, with the package's conventions (xraylarch's)."""
    kt = torch.as_tensor(k, dtype=torch.float64)
    c = ft(torch.as_tensor(spectrum, dtype=torch.float64)[None], kt, window_tensor(kt, FT_WINDOW), FT_WINDOW.kweight)[0]
    return r_grid().numpy(), c.abs().numpy()


def two_panels(curves, title):
    """k² chi(k) and |chi(R)| of several spectra: (label, spectrum, color, line width)."""
    fig, (a, b) = plt.subplots(1, 2, figsize=(11, 3.8))
    shown = (k >= kmin) & (k <= kmax)
    for label, spectrum, color, width in curves:
        a.plot(k[shown], (k ** 2 * spectrum)[shown], color=color, lw=width, label=label)
        b.plot(*chi_r(spectrum), color=color, lw=width, label=label)
    a.set(xlabel="k (1/Å)", ylabel="k² χ(k) (1/Å²)")
    b.set(xlim=(0, 7), ylim=(0, None), xlabel="R (Å)", ylabel="|χ(R)| (1/Å³)")
    fig.legend(*a.get_legend_handles_labels(), loc="lower center", ncol=len(curves), fontsize=9)
    fig.suptitle(title, fontsize=10)
    fig.tight_layout(rect=(0, 0.07, 1, 1))


two_panels([("EvAX's spectrum of its final atoms", evax_fit, ORANGE, 2.2), ("this package, the same atoms", ours_total, BLUE, 1.2)],
           "The same atoms through the two programs")

Single scattering, which is most of the signal, agrees closely between the two programs for the same atoms. The
difference that remains is mostly in the multiple scattering. There the two programs work differently: EvAX
computes a few tens of representative paths and assigns every path of every atom to one of them; this package
computes every path of every atom from its own geometry (tens of thousands of paths for this structure). The
forward model of this package is checked against FEFF 8.5L itself ([Validation](https://github.com/nmarcella/feffGPU/blob/main/docs/validation/index.md)).
Which share of the difference comes from the grouping of paths and which from details of the potentials was not
traced here.

## 7. The fit

**What the literal translation costs.** `search.toml` asks for 64 structures of 256 atoms, each evaluated with all
its multiple-scattering paths in every generation, for thousands of generations. On the development GPU one
generation of 8 structures takes about 30 seconds, so the literal run takes days. (The output files of the EvAX
run are dated six days after its input files.) The cell below can run a few generations of it to see it work
(`LITERAL_GENERATIONS`); it is switched off.

In [ ]:
LITERAL_GENERATIONS = 0        #@param {type:"integer"}

if LITERAL_GENERATIONS:
    short = search_toml.replace(f"generations = {int(evax['Froze_in'])}", f"generations = {LITERAL_GENERATIONS}")
    short = short.replace(f"states = {int(evax['Number_of_states'])}", "states = 8")
    (WORK / "search_short.toml").write_text(short)
    !rmc-exafs run cu_evax/search_short.toml

**The route of this notebook: gradient refinement.** For a structure of one element, `rmc-exafs refine` moves all
atoms together along the gradient of the misfit, with every single- and multiple-scattering path in the gradient,
and takes a step only when the misfit does not rise ([Gradient refinement](https://github.com/nmarcella/feffGPU/blob/main/docs/rmc/refinement.md)). It
needs hundreds of steps where a random search needs many thousands of generations.

What it shares with EvAX's run: the start (the ideal lattice with small random displacements), one copy of the
256-atom cell, the potentials, the cutoff and the filter of the paths, S0² and the energy shift (held).

What differs:

- **The misfit** is the R-factor, in R space, with a Hanning window over the same k range (`kmin = 3.5`,
  `kmax = 12.5`, `dk = 1`: the window reaches from 3 to 13 1/Å), not the wavelet misfit.
- **No limit on the displacements.** EvAX keeps every atom within 0.4 Å of its site. The refinement has no such
  limit, only a hard core that keeps atoms from coming closer than `core_distance`.
- **The convention of S0² and the energy shift** (step 4).

`STEPS` sets the length of the fit: about 12 seconds per step on the development GPU.

In [ ]:
STEPS = int(os.environ.get("FEFFGPU_TUTORIAL_STEPS", 150))     #@param {type:"integer"}

refine_toml = f"""# the gradient refinement of the same problem
[run]
output = "refine"
seed = 1

{SAMPLE}  window = {{ kmin = {kmin + 0.5}, kmax = {kmax - 0.5}, dk = 1.0, kweight = {kweight}, rmin = {rmin}, rmax = {rmax} }}

[amplitude]
s02 = {s02}
de0 = {de0}

[refine]
copies = 1                     # one cell of 256 atoms, as one state of EvAX
steps = {STEPS}
step_size = 0.003              # root-mean-square move of the atoms per step (A)
start_kick = 0.03              # random displacement of every coordinate at the start (A)
core_distance = 2.1            # atoms do not come closer than this (A)
value_every = 5                # steps between full passes of the multiple scattering
log_every = 25
"""
(WORK / "refine.toml").write_text(refine_toml)
print(refine_toml)
!rmc-exafs validate cu_evax/refine.toml

In [ ]:
import json

finished = WORK / "refine" / "Cu_foil" / "refine.json"
if finished.is_file() and json.loads(finished.read_text())["settings"]["steps"] == STEPS:
    print(f"a refinement of {STEPS} steps is already in cu_evax/refine; delete that folder to run it again")
else:
    !rmc-exafs refine cu_evax/refine.toml

## 8. Compare the two fits

The two structures are now scored with the same measures, both through the forward model of this package:

- **R-factor, this package's convention:** the measured spectrum as it is, S0² and the energy shift on the
  calculated spectrum. This is what the refinement minimized.
- **EvAX's misfit:** against EvAX's target, in wavelet space. This is what EvAX minimized, with its own forward
  model; the last line gives EvAX's own number for its structure.

In [ ]:
from feffgpu.exafs import read_chi, to_grid

result = np.load(WORK / "refine" / "Cu_foil" / "refine.npz")
report = json.loads((WORK / "refine" / "Cu_foil" / "refine.json").read_text())
our_atoms = result["positions"]
print(f"refinement: {report['settings']['steps']} steps in {report['seconds'] / 60:.1f} min; "
      f"R-factor {report['misfit_start']:.3f} at the start, {report['misfit_final']:.4f} at the end")

kd, cd = read_chi(WORK / "foils_not_aligned.chi", k_column, chi_column)
data_frame = GpuRmc(phase, k, to_grid(k, kd, cd) / s02, de0, CELL, window=(kmin + 0.5, kmax - 0.5), r_range=(rmin, rmax),
                    rmax=rpath, critpw=criteria, r_min=1.5, dk=1.0)
spectra = {}
print(f"\n{'':34s}{'R-factor':>10s}{'EvAX misfit':>14s}")
for name, atoms in (("EvAX's final atoms", evax_atoms), ("this package's refinement", our_atoms)):
    atoms = np.asarray(atoms, float).reshape(-1, len(ideal), 3)
    _, r_value = data_frame.spectrum_only(atoms)
    spectra[name], _ = spectrum_of(atoms)
    print(f"{name:34s}{r_value:10.4f}{wavelet_misfit(spectra[name]):14.4f}")
print(f"{'EvAX, its own forward model':34s}{'':10s}{wavelet_misfit(evax_fit):14.4f}")

In [ ]:
two_panels([("EvAX's target: the data divided by S0², shifted", evax_target, INK, 2.4), ("EvAX's fit", evax_fit, ORANGE, 1.3),
            ("this package's refinement", spectra["this package's refinement"], BLUE, 1.3)],
           "The measured spectrum and the two fits")

**The structures.** An RMC fit gives atoms, not shell parameters. The numbers of a neighbor shell are counted from
the atoms: the number of neighbors N, their mean distance R, the variance of the distance σ² (the mean-square
relative displacement) and the third cumulant C3, which measures the asymmetry. N is not a result of either fit:
every atom stays near its site, so N is that of the copper lattice.

In [ ]:
def pair_distances(atoms, r_max=6.5):
    """Every distance below r_max between the atoms of a structure (copies, atoms, 3), with periodic images."""
    found = []
    for copy in np.asarray(atoms, float).reshape(-1, len(ideal), 3):
        d = copy[:, None] - copy[None]
        d -= CELL * np.round(d / CELL)
        r = np.sqrt((d ** 2).sum(-1))
        found.append(r[(r > 0.1) & (r < r_max)])
    return np.concatenate(found), len(found) * len(ideal)


SHELLS = ((2.0, 3.1), (3.1, 4.0), (4.0, 4.8), (4.8, 5.4))
structures = (("EvAX", evax_atoms, ORANGE), ("this package", our_atoms, BLUE))

fig, ax = plt.subplots(figsize=(8, 3.6))
for name, atoms, color in structures:
    r, n_atoms = pair_distances(atoms)
    counts, edges = np.histogram(r, bins=np.arange(2.0, 6.5, 0.02))
    ax.step(edges[:-1], counts / n_atoms / 0.02, where="post", color=color, lw=1.4, label=name)
ax.set(xlabel="distance (Å)", ylabel="neighbors per atom per Å")
ax.legend(title="fitted structure")
fig.tight_layout()

print(f"{'shell (Å)':12s}{'':14s}{'N':>7s}{'R (Å)':>9s}{'σ² (Å²)':>10s}{'C3 (Å³)':>11s}")
for lo, hi in SHELLS:
    for name, atoms, _ in structures:
        r, n_atoms = pair_distances(atoms)
        shell = r[(r > lo) & (r < hi)]
        print(f"{lo:.1f} to {hi:.1f}  {name:14s}{len(shell) / n_atoms:7.2f}{shell.mean():9.4f}{shell.var():10.5f}"
              f"{((shell - shell.mean()) ** 3).mean():11.1e}")

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3.4))
sites = ideal.positions
for name, atoms, color in structures:
    reference = evax_sites if name == "EvAX" else sites
    d = np.asarray(atoms, float).reshape(-1, len(ideal), 3) - reference[None]
    d -= CELL * np.round(d / CELL)
    moved = np.sqrt((d ** 2).sum(-1)).ravel()
    ax.hist(moved, bins=np.arange(0, 0.62, 0.02), histtype="step", color=color, lw=1.5, label=f"{name}: largest {moved.max():.2f} Å")
ax.axvline(float(evax["Maximal_displacement"]), color=GRAY, lw=1, ls=":", label="the limit in EvAX's run")
ax.set(xlabel="distance of an atom from its lattice site (Å)", ylabel="atoms")
ax.legend()
fig.tight_layout()

## 9. How to read the comparison

- **Two fits of one spectrum are two members of a family.** Many arrangements of 256 atoms give nearly the same
  spectrum. The fitted spectra can agree far better than the shell numbers do. The spread between independent fits
  (other seeds, other starts) is the uncertainty of an RMC result; one fit does not show it
  ([Uncertainty](https://github.com/nmarcella/feffGPU/blob/main/docs/rmc/uncertainty.md)).
- **S0² trades with the disorder.** Both fits hold S0² at the value of EvAX's parameter file. A value that is too
  large for the potentials in use can be compensated by atoms far from their sites, which adds weight to the tails
  of the shells and shows as a large σ² and a large C3. `rmc-exafs run` can fit S0² and the energy shift during
  the search (`[amplitude] mode = "fit"`).
- **The limits differ.** EvAX's atoms stay within 0.4 Å of their sites; the refinement has no such limit. The last
  figure shows whether that mattered.
- **A longer fit is one number.** Raise `STEPS`, or continue with the population search and its constraints
  ([The search](https://github.com/nmarcella/feffGPU/blob/main/docs/rmc/search.md)).

When you publish results obtained with this package, cite FEFF and EvAX ([Citing](https://github.com/nmarcella/feffGPU/blob/main/docs/citing.md)).

### Keep the results (Colab)

The next cell packs the working folder (without EvAX's files) and downloads it. Outside Colab it does nothing.

In [ ]:
if IN_COLAB:
    from google.colab import files
    for name in ("Cu.p1", "foils_not_aligned.chi"):
        (WORK / name).unlink(missing_ok=True)
    shutil.make_archive("cu_evax_results", "zip", WORK)
    files.download("cu_evax_results.zip")